# HeptaAI Data Profiling Demo

This notebook demonstrates HeptaAI's data profiling features for quick dataset exploration before running quality checks.

**What you'll learn:**
- Quick dataset profiling with `profile_dataset()`
- Train vs test comparison with `compare_datasets()`
- Complete workflow: Profile → Stats → Detect → Fix
- Real-world use cases and best practices

In [ ]:
# Setup - add parent directory to path
import sys
sys.path.insert(0, '..')

import heptaai as hepta
import pandas as pd
import numpy as np

print(f"HeptaAI version: {hepta.__version__}")

## 1. Basic Dataset Profiling

Let's start by profiling a single dataset to understand what we're working with.

In [ ]:
# Profile the clean MovieLens training data
hepta.profile_dataset(
    "../playground/raw_data/movielens_train.csv",
    label_col="label"
)

**Key insights from the profile:**
- ✅ Dataset size and shape
- ✅ Column types (numeric vs categorical)
- ✅ Missing value summary
- ✅ Label distribution and balance assessment
- ✅ Sample preview
- ✅ Automatic quality warnings

All in **< 1 second**!

## 2. Profiling Anomalous Data

Now let's profile a dataset with injected quality issues.

In [ ]:
# Profile the anomalous dataset
hepta.profile_dataset(
    "../playground/raw_data/movielens_anomalous.csv",
    label_col="label"
)

**Notice the differences:**
- ⚠️ Missing values detected (5% in genre column)
- ⚠️ Quick assessment flags potential issues

The profile immediately tells you there are quality issues without running expensive statistics!

## 3. Comparing Train vs Test Datasets

One of the most important sanity checks: are train and test distributions similar?

In [ ]:
# Compare train vs test side-by-side
hepta.compare_datasets(
    train_data="../playground/raw_data/movielens_train.csv",
    test_data="../playground/raw_data/movielens_test.csv",
    label_col="label"
)

**What to look for:**
- **Row count delta**: Large differences might indicate sampling bias
- **Column count**: Should be identical (schema consistency)
- **Missing value delta**: Sudden increase in test could indicate data drift
- **Label distribution shift**: >10% difference is concerning

This catches issues like:
- ❌ Train on 2020 data, test on 2021 data → temporal drift
- ❌ Different preprocessing pipelines
- ❌ Data leakage (test is cleaner than train)

## 4. Profiling In-Memory DataFrames

You don't need to save to disk first - profile DataFrames directly!

In [ ]:
# Create a synthetic dataset with quality issues
np.random.seed(42)

n_samples = 10000

df = pd.DataFrame({
    'user_id': range(n_samples),
    'age': np.random.randint(18, 70, n_samples),
    'country': np.random.choice(['US', 'UK', 'FR', 'DE'], n_samples),
    'session_duration': np.random.exponential(300, n_samples),
    'page_views': np.random.poisson(5, n_samples),
    'converted': np.random.binomial(1, 0.03, n_samples),  # 3% conversion (imbalanced!)
})

# Inject missing values
df.loc[np.random.choice(n_samples, 500, replace=False), 'session_duration'] = np.nan

# Inject duplicates
df = pd.concat([df, df.sample(200)], ignore_index=True)

print(f"Created synthetic dataset: {df.shape}")
print(f"True duplicate rate: {df.duplicated().sum() / len(df):.1%}")

In [ ]:
# Profile the in-memory DataFrame
hepta.profile_dataset(df, label_col="converted")

**Automatic detection:**
- ⚠️ Severe class imbalance (3% conversion rate)
- ⚠️ Missing values in session_duration
- ⚠️ High duplicate rate

All caught before you waste time training a model!

## 5. Complete Workflow: Profile → Stats → Detect

Recommended workflow for production data quality checks.

In [ ]:
print("="*70)
print("Step 1: Quick Profile (0.5 seconds)")
print("="*70)

hepta.profile_dataset(
    "../playground/raw_data/movielens_anomalous.csv",
    label_col="label"
)

In [ ]:
print("\n" + "="*70)
print("Step 2: Detailed Statistics (2-5 seconds)")
print("="*70 + "\n")

stats = hepta.generate_statistics(
    "../playground/raw_data/movielens_anomalous.csv",
    label_col="label"
)

print(f"✅ Generated statistics for {stats.n_rows:,} rows, {len(stats.features)} features")
print(f"   Duplicate rate: {stats.duplicate_rate:.1%}")
print(f"   Label entropy: {stats.label_entropy:.3f}")

In [ ]:
print("\n" + "="*70)
print("Step 3: Issue Detection (< 1 second)")
print("="*70 + "\n")

issues = hepta.detect_issues(stats)

print(f"✅ Ran 6 detectors")
print(f"   Found {len(issues)} issue(s)\n")

hepta.display_issues(issues)

## 6. Real-World Use Case: Daily Data Pipeline Monitoring

Simulate monitoring a daily data refresh.

In [ ]:
# Simulate "yesterday's" data (clean)
df_yesterday = pd.read_csv("../playground/raw_data/movielens_train.csv")

# Simulate "today's" data (with a pipeline bug)
df_today = df_yesterday.copy()

# Bug 1: Half the data is missing (pipeline timeout)
df_today = df_today.sample(frac=0.5, random_state=42)

# Bug 2: Label column got corrupted (all zeros)
df_today['label'] = 0

# Bug 3: New missing values appeared
df_today.loc[df_today.sample(frac=0.1, random_state=42).index, 'genre'] = np.nan

print("Simulated pipeline bug:")
print(f"  Yesterday: {len(df_yesterday):,} rows")
print(f"  Today:     {len(df_today):,} rows")
print(f"\n  Label distribution changed:")
print(f"  Yesterday: {df_yesterday['label'].mean():.1%} positive")
print(f"  Today:     {df_today['label'].mean():.1%} positive")

In [ ]:
# Compare to detect the pipeline issue
hepta.compare_datasets(
    train_data=df_yesterday,
    test_data=df_today,
    label_col="label"
)

**🚨 Pipeline issues caught:**
1. ⚠️ Large difference in row count (-50%) → Data pipeline timeout
2. ⚠️ Label distribution shift (48% → 0%) → Label corruption
3. ⚠️ Missing value rate increased (+10%) → Data quality degradation

Without profiling, you might have trained a model on this corrupted data!

## 7. Performance Comparison

How fast is profiling vs full statistics?

In [ ]:
import time

# Create a larger dataset for benchmarking
large_df = pd.concat([df_yesterday] * 10, ignore_index=True)
print(f"Dataset size: {len(large_df):,} rows × {len(large_df.columns)} columns")
print(f"Memory: {large_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print()

In [ ]:
# Benchmark profiling (fast)
start = time.time()
hepta.profile_dataset(large_df, label_col="label")
profile_time = time.time() - start

print(f"\n⏱️  Profiling time: {profile_time:.3f} seconds")

In [ ]:
# Benchmark full statistics (slower)
start = time.time()
stats = hepta.generate_statistics(large_df, label_col="label")
stats_time = time.time() - start

print(f"⏱️  Full statistics time: {stats_time:.3f} seconds")
print(f"\n💨 Profiling is {stats_time/profile_time:.1f}× faster!")

## 8. When to Use What?

| Scenario | Use `profile_dataset()` | Use `generate_statistics()` |
|----------|------------------------|----------------------------|
| First time seeing dataset | ✅ Yes | ⏭️ Later |
| Quick sanity check | ✅ Yes | ❌ Overkill |
| Compare train/test | ✅ Yes | ⏭️ Optional |
| Daily pipeline monitoring | ✅ Yes | ⏭️ Weekly |
| Before model training | ✅ Yes | ✅ Yes (both) |
| Need per-feature histograms | ❌ No | ✅ Yes |
| Need percentiles (p25, p75, p99) | ❌ No | ✅ Yes |
| Need train-test skew detection | ❌ No | ✅ Yes |

**Recommendation:**
1. Always start with `profile_dataset()` (< 1 second)
2. If issues found, run `generate_statistics()` for details
3. Use `compare_datasets()` before every training run

## 9. Summary

**What we learned:**

✅ **Quick profiling** catches issues in < 1 second
- Missing values
- Class imbalance
- Duplicates
- Schema issues

✅ **Dataset comparison** prevents training on bad data
- Distribution shifts
- Pipeline bugs
- Label corruption

✅ **Complete workflow** ensures data quality
- Profile → Stats → Detect → Fix
- Fast iteration
- Proactive quality checks

**Next steps:**
- Try profiling your own datasets
- Add profiling to your data pipelines
- Use `compare_datasets()` in CI/CD

**Coming in v0.2:**
- `generate_manifest()` - Automatic data cleaning
- `manifest.apply()` - Apply fixes to your data
- Quality score (0-100) per dataset

## 10. Try It Yourself!

Use the cells below to experiment with your own data.

In [ ]:
# Your dataset here
# hepta.profile_dataset("your_data.csv", label_col="your_label")

In [ ]:
# Compare your train vs test
# hepta.compare_datasets(
#     train_data="your_train.csv",
#     test_data="your_test.csv",
#     label_col="your_label"
# )